In [1]:
# N-1 — Environment. Afterwards: Runtime → Restart session, then run N-2 onward.
!apt-get -qq install -y git-lfs > /dev/null
import subprocess
from pathlib import Path
ROOT = Path("/content/Project-phase-2")
if not ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/CMU-18734-17731-2026-Fall/Project-phase-2.git", str(ROOT)], check=True)
subprocess.run(["git", "lfs", "install"], check=True)
subprocess.run(["git", "lfs", "pull"], cwd=ROOT, check=True)
%pip install -q "transformers==4.56.2" "peft==0.13.2" "accelerate==1.10.1" "datasets==4.1.1"
print("Done. Restart the session now.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requ

In [3]:
# N-2 — Imports and constants. Nothing inherited from earlier runtimes.
import os, sys, gc, json, random, shutil, hashlib, subprocess
os.environ.update(USE_TF="0", USE_FLAX="0", TOKENIZERS_PARALLELISM="false")
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, roc_curve
import transformers, peft
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
assert transformers.__version__ == "4.56.2" and peft.__version__ == "0.13.2"
assert torch.cuda.is_available(), "GPU runtime required"

from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/Project-phase-2")
DRIVE = Path("/content/drive/MyDrive/MIA_phase2")
OLD = DRIVE / "cache"
MODEL_NAME, MAX_LENGTH = "gpt2_3_lora32_adamw_b8_lr2", 512
PHASES = ("train", "val", "final")                    # "train" = the dev target's evaluation texts
PAIRS = [("inA", "inB"), ("p2A", "p2B"), ("p3A", "p3B"), ("p4A", "p4B")]
NEW_PAIRS = [("p5A", "p5B"), ("p6A", "p6B")]
NEW_PAIR_NO = {("p5A", "p5B"): 5, ("p6A", "p6B"): 6}
REF_NAMES = [n for p in PAIRS for n in p]
KAPPA = 8
device, model_dtype = torch.device("cuda"), torch.float16
env = os.environ.copy()
assert ROOT.exists() and DRIVE.exists()
print("OK")

Mounted at /content/drive
OK


In [4]:
# N-3 — Texts in original order, labels, tokenizer, membership, global evaluation groups.
def read_json(p):
    with Path(p).open("r", encoding="utf-8") as f:
        return json.load(f)

TEXTS = {ph: read_json(ROOT / "data" / ph / "test.json") for ph in PHASES}
LABELS = np.asarray(read_json(ROOT / "data" / "train" / "test_label.json"), dtype=np.int64)
assert all(len(TEXTS[ph]) == 2000 for ph in PHASES) and LABELS.sum() == 1000
DEV_ORDER_SHA = hashlib.sha256("\x1e".join(TEXTS["train"]).encode()).hexdigest()

ADAPTER_DIR = ROOT / "models" / "train" / MODEL_NAME
adapter_config = read_json(ADAPTER_DIR / "adapter_config.json")
BASE_MODEL_NAME = adapter_config["base_model_name_or_path"]
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
SHADOW_TRAIN_SIZE = len(read_json(ROOT / "data" / "train" / "train_finetune.json"))
assert SHADOW_TRAIN_SIZE == 10000

norm = lambda t: " ".join(t.split())
def prefix_keys(texts):
    keys = []
    for s in range(0, len(texts), 256):
        for ids in tokenizer(texts[s:s + 256], add_special_tokens=False, truncation=True,
                             max_length=MAX_LENGTH, padding=False)["input_ids"]:
            keys.append(hashlib.sha256(np.asarray(ids, dtype="<i4").tobytes()).hexdigest())
    return keys
EVAL_NORM = {ph: [norm(t) for t in TEXTS[ph]] for ph in PHASES}
EVAL_PREF = {ph: prefix_keys(TEXTS[ph]) for ph in PHASES}

def membership(train_texts):
    ns = {norm(t) for t in train_texts}; ps = set(prefix_keys(list(train_texts)))
    return {ph: np.array([(a in ns) or (b in ps) for a, b in zip(EVAL_NORM[ph], EVAL_PREF[ph])]) for ph in PHASES}
def shadow_texts(n):
    return [r if isinstance(r, str) else r["text"] for r in read_json(DRIVE / "data" / f"shadow_{n}" / "train_finetune.json")]

MEM = {n: membership(shadow_texts(n)) for n in REF_NAMES}
for a, b in PAIRS[1:]:
    for ph in PHASES:
        assert (MEM[a][ph] ^ MEM[b][ph]).all(), (a, b, ph)
assert int((MEM["inA"]["train"] & MEM["inB"]["train"]).sum()) == 7

# Global groups over dev+val+final (same item order and keys as the original construction).
EVAL_ITEMS = [(ph, i) for ph in PHASES for i in range(2000)]
parent = list(range(len(EVAL_ITEMS)))
def _find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
first = {}
for k, (ph, i) in enumerate(EVAL_ITEMS):
    for key in (("n", EVAL_NORM[ph][i]), ("p", EVAL_PREF[ph][i])):
        if key in first: parent[_find(k)] = _find(first[key])
        else: first[key] = k
GROUPS = {}
for k in range(len(EVAL_ITEMS)):
    GROUPS.setdefault(_find(k), []).append(k)
EVAL_NORM_SET = {EVAL_NORM[ph][i] for ph, i in EVAL_ITEMS}
EVAL_PREF_SET = {EVAL_PREF[ph][i] for ph, i in EVAL_ITEMS}

def assign_sides(pair_no):
    rng = np.random.default_rng(7000 + pair_no)
    side_items, side_n = ([], []), [0, 0]
    for g in rng.permutation(sorted(GROUPS)):
        g = int(g)
        s = int(rng.integers(2)) if side_n[0] == side_n[1] else int(side_n[1] < side_n[0])
        side_items[s].extend(GROUPS[g]); side_n[s] += len(GROUPS[g])
    return side_items, side_n

# Reproduction check: the assignment rule must regenerate pairs 2–4 exactly from their saved datasets.
for k in (2, 3, 4):
    side_items, _ = assign_sides(k)
    flag = np.zeros(len(EVAL_ITEMS), bool); flag[side_items[1]] = True
    got = np.concatenate([MEM[f"p{k}B"][ph] for ph in PHASES])
    assert np.array_equal(flag, got), f"group assignment does not reproduce pair {k}"
print(f"{len(EVAL_ITEMS)} eval items in {len(GROUPS)} groups; assignment rule reproduces pairs 2–4.")

6000 eval items in 5981 groups; assignment rule reproduces pairs 2–4.


In [5]:
# N-4 — Cached losses, alignment checks, frozen LR4 reproduction, bootstrap indices.
def split_npz(path, key="losses"):
    z = np.load(path)
    return [a.astype(np.float64) for a in np.split(z[key], np.cumsum(z["lengths"])[:-1])]

TGT = {"train": split_npz(OLD / "train_base_target_losses.npz", "target_losses"),
       "val":   split_npz(OLD / "val_ref_train_losses.npz"),
       "final": split_npz(OLD / "final_train_tgt_losses.npz")}
REF = {n: {ph: split_npz(OLD / f"{ph}_shadow_{n}_losses.npz") for ph in PHASES} for n in REF_NAMES}
for ph in PHASES:
    exp_len = np.array([len(ids) - 1 for ids in tokenizer(TEXTS[ph], truncation=True, max_length=MAX_LENGTH)["input_ids"]])
    assert np.array_equal(exp_len, [a.size for a in TGT[ph]]), ph
    for n in REF_NAMES:
        assert all(r.shape == t.shape for r, t in zip(REF[n][ph], TGT[ph])), (n, ph)

fixed = [split_npz(OLD / "train_shadow_1001_losses.npz", "shadow_losses"), split_npz(OLD / "train_shadow_1002_losses.npz"),
         split_npz(OLD / "train_ref_val_losses.npz"), split_npz(OLD / "train_ref_final_losses.npz")]
FIX4 = np.stack([np.array([a.mean() for a in f]) for f in fixed])   # shadow1001, shadow1002, val tgt, final tgt
DEV_MT = np.array([a.mean() for a in TGT["train"]])
DEV_L = np.array([a.size for a in TGT["train"]], dtype=np.float64)
mean_loss = lambda n: np.array([a.mean() for a in REF[n]["train"]])

def tpr_strict(y, s):
    fpr, tpr, _ = roc_curve(y, s)
    return float(np.max(tpr[fpr < 0.01]))
def strict_fast(s, y):
    neg, pos = s[y == 0], s[y == 1]
    return np.mean(pos > np.sort(neg)[::-1][int(np.ceil(0.01 * neg.size)) - 1])

def lr_frozen(mt, L, P, Pin, kappa=KAPPA):
    """Frozen LR4: unequal-variance Gaussian LR on √T·mean loss, κ-shrinkage to pooled IN/OUT variances, ×(1+1/n)."""
    sq = np.sqrt(L); U, u = P * sq, mt * sq
    fin = np.isfinite(P); m_in, m_out = Pin & fin, ~Pin & fin
    n_in, n_out = m_in.sum(0), m_out.sum(0)
    assert (n_in > 0).all() and (n_out > 0).all()
    Uz = np.where(fin, U, 0.0)
    mu_in = np.where(m_in, Uz, 0.0).sum(0) / n_in
    mu_out = np.where(m_out, Uz, 0.0).sum(0) / n_out
    rss_in = np.where(m_in, (Uz - mu_in) ** 2, 0.0).sum(0)
    rss_out = np.where(m_out, (Uz - mu_out) ** 2, 0.0).sum(0)
    df_in, df_out = np.maximum(n_in - 1, 0), np.maximum(n_out - 1, 0)
    pool_out = rss_out.sum() / df_out.sum()
    pool_in = rss_in.sum() / df_in.sum() if df_in.sum() > 0 else pool_out
    s_in = (rss_in + kappa * pool_in) / (df_in + kappa) * (1 + 1 / n_in)
    s_out = (rss_out + kappa * pool_out) / (df_out + kappa) * (1 + 1 / n_out)
    A = (-(u - mu_in) ** 2 / (2 * s_in) - 0.5 * np.log(s_in)) - (-(u - mu_out) ** 2 / (2 * s_out) - 0.5 * np.log(s_out))
    assert np.isfinite(A).all()
    return A

S4 = np.load(OLD / "staged4_dev_scores_4pairs.npz")
LR4_DEV = S4["LR (all OUT, κ=8 fixed)"].astype(np.float64)
P_lr4 = np.vstack([np.stack([mean_loss(n) for n in REF_NAMES]), FIX4])
PIN_lr4 = np.vstack([np.stack([MEM[n]["train"] for n in REF_NAMES]), np.zeros(FIX4.shape, bool)])
assert np.allclose(lr_frozen(DEV_MT, DEV_L, P_lr4, PIN_lr4), LR4_DEV, rtol=0, atol=1e-6), "LR4 not reproduced"
assert np.isclose(tpr_strict(LABELS, LR4_DEV), read_json(DRIVE / "frozen_LR4_k8" / "settings.json")["dev"]["LR"])

# Same STAGED-5 mock numbers as before (pairs 1–2 refs + shadows 1001/1002), as a second reproduction check.
R0 = ["inA", "inB", "p2A", "p2B"]
P0 = np.vstack([np.stack([mean_loss(n) for n in R0]), FIX4[:2]])
PIN0 = np.vstack([np.stack([MEM[n]["train"] for n in R0]), np.zeros((2, 2000), bool)])
for t, exp in (("p3A", 0.8483), ("p3B", 0.8084), ("p4A", 0.7450), ("p4B", 0.6586)):
    got = tpr_strict(MEM[t]["train"].astype(int), lr_frozen(mean_loss(t), DEV_L, P0, PIN0))
    assert abs(got - exp) < 6e-4, (t, got, exp)

rng = np.random.default_rng(20261017)                         # same dev bootstrap as earlier notebooks
pos_i, neg_i = np.flatnonzero(LABELS == 1), np.flatnonzero(LABELS == 0)
boot = [np.concatenate([rng.choice(pos_i, pos_i.size), rng.choice(neg_i, neg_i.size)]) for _ in range(2000)]
print(f"Frozen LR4 reproduced: AUC={roc_auc_score(LABELS, LR4_DEV):.4f} | strict TPR={tpr_strict(LABELS, LR4_DEV):.4f}; "
      "STAGED-5 mock values reproduced.")

Frozen LR4 reproduced: AUC=0.9744 | strict TPR=0.7120; STAGED-5 mock values reproduced.


In [8]:
# Remount Drive and reopen the lazily-loaded LR4 file.
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception as e:
    print("unmount:", e)
drive.mount("/content/drive", force_remount=True)
S4 = np.load(OLD / "staged4_dev_scores_4pairs.npz")
assert np.allclose(S4["LR (all OUT, κ=8 fixed)"].astype(np.float64), LR4_DEV, rtol=0, atol=0)
print("Drive remounted; S4 reopened.")

Mounted at /content/drive
Drive remounted; S4 reopened.


In [11]:
# E-0 — Ensemble setup. Run after N-2..N-4. CPU is enough.
from scipy.special import logsumexp
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

FROZEN_PAIRS = [("inA", "inB"), ("p2A", "p2B"), ("p3A", "p3B"), ("p4A", "p4B")]     # explicit; not inherited
FROZEN_NAMES = [n for p in FROZEN_PAIRS for n in p]
PARTNER = {a: b for a, b in FROZEN_PAIRS} | {b: a for a, b in FROZEN_PAIRS}
FRESH_AUX = {"p4A", "p4B"}
C_FIXED, TRIM, N_TEXT_FOLDS, N_BOOT = 0.01, 0.2, 2, 1000                         # fixed in advance
ENS_CACHE = DRIVE / "cache" / "ensemble"; ENS_CACHE.mkdir(parents=True, exist_ok=True)
EZ_CACHE = DRIVE / "cache" / "ezmia"

ML = {n: np.array([a.mean() for a in REF[n]["train"]]) for n in FROZEN_NAMES}
MEMD = {n: MEM[n]["train"].copy() for n in FROZEN_NAMES}
SQ = np.sqrt(DEV_L)

def log_trim_mean_exp(L, prop=TRIM):
    K = L.shape[0]; c = int(prop * K)
    S = np.sort(L, axis=0)[c:K - c]
    return logsumexp(S, axis=0) - np.log(S.shape[0])

def b_ext(mt, P, Pin):
    out = ~Pin & np.isfinite(P); n_out = out.sum(0)
    assert (n_out > 0).all()
    return (np.where(out, P, 0.0).sum(0) / n_out - mt) * SQ, n_out

def features(mt, excl):
    """LR, B_ext and RMIA a(x) from one reference pool: frozen paired models minus `excl`, plus the 4 fixed OUT refs."""
    paired = [n for n in FROZEN_NAMES if n not in set(excl)]
    P = np.vstack([np.stack([ML[n] for n in paired]), FIX4])
    Pin = np.vstack([np.stack([MEMD[n] for n in paired]), np.zeros(FIX4.shape, bool)])
    B, n_out = b_ext(mt, P, Pin)
    feats = {"LR": lr_frozen(mt, DEV_L, P, Pin),               # pooled variances recomputed for this pool
             "B_ext": B,                                       # same eligible OUT rows as LR
             "RMIA_a": -mt - log_trim_mean_exp(-np.stack([ML[n] for n in paired]))}
    for k, v in feats.items():
        assert np.isfinite(v).all(), k
    n_in = Pin.sum(0)
    info = {"paired refs": ",".join(paired), "LR refs": P.shape[0], "IN refs min–max": f"{n_in.min()}–{n_in.max()}",
            "OUT refs min–max": f"{n_out.min()}–{n_out.max()}", "RMIA refs": len(paired),
            "RMIA trimmed per side": int(TRIM * len(paired))}
    return feats, info, set(paired)

# Checks: real-target features with no exclusions reproduce LR4, B5 (pair-1 pool) and the saved RMIA a(x).
REAL, REAL_INFO, _ = features(DEV_MT, ())
assert np.allclose(REAL["LR"], LR4_DEV, rtol=0, atol=1e-6)
P_b5 = np.vstack([np.stack([ML[n] for n in ("inA", "inB")]), FIX4])
Pin_b5 = np.vstack([np.stack([MEMD[n] for n in ("inA", "inB")]), np.zeros(FIX4.shape, bool)])
assert np.allclose(b_ext(DEV_MT, P_b5, Pin_b5)[0], S4["B5"], atol=1e-4)
z = np.load(DRIVE / "cache" / "rmia" / "rmia_dev_scores.npz")
k_ax = [k for k in z.files if k.startswith("mean") and "a(x)" in k]
assert len(k_ax) == 1, z.files
assert np.allclose(z[k_ax[0]], REAL["RMIA_a"], atol=1e-8)

# Paired stratified bootstrap of the 2,000 real dev examples, defined here.
_rng = np.random.default_rng(20261017)
_pos, _neg = np.flatnonzero(LABELS == 1), np.flatnonzero(LABELS == 0)
ENS_BOOT = [np.concatenate([_rng.choice(_pos, _pos.size), _rng.choice(_neg, _neg.size)]) for _ in range(2000)]
print("Frozen 4-pair list set; LR = LR4, B_ext reproduces B5, RMIA a(x) matches; ENS_BOOT defined.")

Frozen 4-pair list set; LR = LR4, B_ext reproduces B5, RMIA a(x) matches; ENS_BOOT defined.


In [12]:
# E-1 — Features per held-out pair. Training mocks exclude {a, b, m, partner(m)}; held-out targets exclude {a, b}.
FOLD_FEAT, count_rows = {}, []
for p, (a, b) in enumerate(FROZEN_PAIRS):
    d = {"train": {}, "test": {}}
    for m in [n for n in FROZEN_NAMES if n not in (a, b)]:
        d["train"][m], info, used = features(ML[m], {a, b, m, PARTNER[m]})
        assert not used & {a, b, m, PARTNER[m]}
        count_rows.append({"held-out pair": f"{a}/{b}", "role": "training mock", "target": m, **info})
    for m in (a, b):
        d["test"][m], info, used = features(ML[m], {a, b})
        assert not used & {a, b}
        count_rows.append({"held-out pair": f"{a}/{b}", "role": "held-out target", "target": m, **info})
    FOLD_FEAT[p] = d
count_rows.append({"held-out pair": "-", "role": "real dev target", "target": "dev", **REAL_INFO})
counts = pd.DataFrame(count_rows)
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 60)
print(counts.drop(columns="paired refs").to_string(index=False))
counts.to_csv(ENS_CACHE / "ens_fold_reference_counts.csv", index=False)

ez = {m: (EZ_CACHE / f"ezmia_mock_{m}_stats.npz").exists() for m in FROZEN_NAMES}
print("\nEZ own top-1 cached for mock targets:", ez, "| real target:", (EZ_CACHE / "ezmia_dev_target_stats.npz").exists())

held-out pair            role target  LR refs IN refs min–max OUT refs min–max  RMIA refs  RMIA trimmed per side
      inA/inB   training mock    p2A        8             2–2              6–6          4                      0
      inA/inB   training mock    p2B        8             2–2              6–6          4                      0
      inA/inB   training mock    p3A        8             2–2              6–6          4                      0
      inA/inB   training mock    p3B        8             2–2              6–6          4                      0
      inA/inB   training mock    p4A        8             2–2              6–6          4                      0
      inA/inB   training mock    p4B        8             2–2              6–6          4                      0
      inA/inB held-out target    inA       10             3–3              7–7          6                      1
      inA/inB held-out target    inB       10             3–3              7–7          6       

In [11]:
# E-2 — Leave one pair out (features and rows) × 2 global-text-group folds; train-only scaling; fixed C.
dev_root = np.array([_find(k) for k in range(2000)])
perm = np.random.default_rng(20261018).permutation(np.unique(dev_root))
fold_of = {g: j % N_TEXT_FOLDS for j, g in enumerate(perm)}
TFOLD = np.array([fold_of[g] for g in dev_root])
print("Dev texts per text fold:", np.bincount(TFOLD))

FEATSETS = {"LR": ["LR"], "LR+B_ext": ["LR", "B_ext"], "LR+B_ext+RMIA_a": ["LR", "B_ext", "RMIA_a"]}
def design(fd, models, idx, cols):
    X = np.vstack([np.column_stack([fd[m][c][idx] for c in cols]) for m in models])
    y = np.concatenate([MEMD[m][idx].astype(int) for m in models])
    return X, y
def fit(X, y):
    sc = StandardScaler().fit(X)
    return sc, LogisticRegression(C=C_FIXED, max_iter=5000).fit(sc.transform(X), y)

rows, coefs, cv_scores, draws = [], [], {}, {}
for p, (a, b) in enumerate(FROZEN_PAIRS):
    ftr, fte = FOLD_FEAT[p]["train"], FOLD_FEAT[p]["test"]
    for f in range(N_TEXT_FOLDS):
        tr, te = np.flatnonzero(TFOLD != f), np.flatnonzero(TFOLD == f)
        fitted = {}
        for fs, cols in FEATSETS.items():
            sc, clf = fit(*design(ftr, list(ftr), tr, cols))
            fitted[fs] = (sc, clf, cols)
            coefs.append({"held_out": f"{a}/{b}", "text_fold": f, "featset": fs, "intercept": clf.intercept_[0],
                          **{f"w_{c}": w for c, w in zip(cols, clf.coef_[0])}})
        assert fitted["LR"][1].coef_[0][0] > 0
        for m in (a, b):
            y = MEMD[m][te].astype(int)
            pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
            rng = np.random.default_rng(1000 + 10 * FROZEN_NAMES.index(m) + f)
            bidx = [np.concatenate([rng.choice(pos, pos.size), rng.choice(neg, neg.size)]) for _ in range(N_BOOT)]
            for fs, (sc, clf, cols) in fitted.items():
                s = clf.decision_function(sc.transform(design(fte, [m], te, cols)[0]))
                t = tpr_strict(y, s); assert np.isclose(strict_fast(s, y), t)
                if fs == "LR":
                    assert np.isclose(t, tpr_strict(y, fte[m]["LR"][te]))
                cv_scores[f"{m}__fold{f}__{fs.replace('+', '_plus_')}"] = s
                draws[(m, f, fs)] = np.array([strict_fast(s[ix], y[ix]) for ix in bidx])
                rows.append({"target": m, "aux": "fresh" if m in FRESH_AUX else "pooled", "fold": f, "featset": fs,
                             "n texts": te.size, "n non-members": neg.size, "TPR": t, "AUC": roc_auc_score(y, s)})
cv = pd.DataFrame(rows)

summ = []
for m in FROZEN_NAMES:
    for fs in FEATSETS:
        sub, base = cv[(cv.target == m) & (cv.featset == fs)], cv[(cv.target == m) & (cv.featset == "LR")]
        d = np.mean([draws[(m, f, fs)] - draws[(m, f, "LR")] for f in range(N_TEXT_FOLDS)], axis=0)
        lo, hi = np.percentile(d, [2.5, 97.5])
        summ.append({"target": m, "aux": "fresh" if m in FRESH_AUX else "pooled", "featset": fs,
                     "TPR (fold mean)": sub.TPR.mean(), "AUC (fold mean)": sub.AUC.mean(),
                     "Δ TPR vs LR": sub.TPR.mean() - base.TPR.mean(), "95% CI": f"[{lo:+.3f}, {hi:+.3f}]",
                     "P(Δ>0)": float(np.mean(d > 0))})
summ = pd.DataFrame(summ)
print("\nE-2: held-out pair excluded from training rows AND from every feature's reference pool; unseen text groups:")
print(summ.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nAverages by auxiliary-data type:")
print(summ.groupby(["aux", "featset"])[["TPR (fold mean)", "AUC (fold mean)", "Δ TPR vs LR"]].mean()
      .to_string(float_format=lambda x: f"{x:.4f}"))
coef_tab = pd.DataFrame(coefs)
print("\nStandardized coefficients across the 8 fits:")
print(coef_tab.groupby("featset")[[c for c in coef_tab.columns if c.startswith("w_")]].agg(["mean", "min", "max"])
      .to_string(float_format=lambda x: f"{x:+.3f}"))

cv.to_csv(ENS_CACHE / "ens_cv_rows.csv", index=False); summ.to_csv(ENS_CACHE / "ens_cv_summary.csv", index=False)
coef_tab.to_csv(ENS_CACHE / "ens_cv_coefs.csv", index=False)
np.savez(ENS_CACHE / "ens_cv_scores.npz", text_fold=TFOLD, **cv_scores)

Dev texts per text fold: [1000 1000]

E-2: held-out pair excluded from training rows AND from every feature's reference pool; unseen text groups:
target    aux         featset  TPR (fold mean)  AUC (fold mean)  Δ TPR vs LR           95% CI  P(Δ>0)
   inA pooled              LR           0.8512           0.9857       0.0000 [+0.000, +0.000]  0.0000
   inA pooled        LR+B_ext           0.8171           0.9859      -0.0341 [-0.074, +0.021]  0.1560
   inA pooled LR+B_ext+RMIA_a           0.8540           0.9860       0.0028 [-0.048, +0.038]  0.5270
   inB pooled              LR           0.8277           0.9861       0.0000 [+0.000, +0.000]  0.0000
   inB pooled        LR+B_ext           0.8030           0.9853      -0.0247 [-0.080, +0.022]  0.2110
   inB pooled LR+B_ext+RMIA_a           0.8076           0.9850      -0.0201 [-0.191, +0.026]  0.1840
   p2A pooled              LR           0.8480           0.9840       0.0000 [+0.000, +0.000]  0.0000
   p2A pooled        LR+B_ext         

In [12]:
# E-3 — Train on all 8 mock targets × all 2,000 dev texts (shadow labels only); score the real dev target.
# This tests transfer to a NEW MODEL on candidate texts the classifier has seen (with shadow labels).
# It does not test transfer to unseen text groups; that is E-2's additional test.
MOCK_FULL, mock_info = {}, []
for m in FROZEN_NAMES:
    MOCK_FULL[m], info, used = features(ML[m], {m, PARTNER[m]})
    assert not used & {m, PARTNER[m]}
    mock_info.append({"role": "training mock", "target": m, **info})
mock_info.append({"role": "real dev target", "target": "dev", **REAL_INFO})
print(pd.DataFrame(mock_info).drop(columns="paired refs").to_string(index=False))

dev_rows, dev_scores, dev_coefs = [], {}, {}
for fs, cols in FEATSETS.items():
    sc, clf = fit(*design(MOCK_FULL, FROZEN_NAMES, np.arange(2000), cols))
    dev_scores[fs] = clf.decision_function(sc.transform(np.column_stack([REAL[c] for c in cols])))
    dev_coefs[fs] = {"standardized_w": dict(zip(cols, clf.coef_[0].tolist())), "intercept": float(clf.intercept_[0]),
                     "scaler_mean": dict(zip(cols, sc.mean_.tolist())), "scaler_scale": dict(zip(cols, sc.scale_.tolist()))}
assert np.isclose(tpr_strict(LABELS, dev_scores["LR"]), tpr_strict(LABELS, LR4_DEV))

bt = {fs: np.array([strict_fast(s[ix], LABELS[ix]) for ix in ENS_BOOT]) for fs, s in dev_scores.items()}
for fs, s in dev_scores.items():
    d = bt[fs] - bt["LR"]; lo, hi = np.percentile(d, [2.5, 97.5])
    dev_rows.append({"featset": fs, "dev AUC": roc_auc_score(LABELS, s), "strict TPR": tpr_strict(LABELS, s),
                     "Δ TPR vs LR (= LR4)": tpr_strict(LABELS, s) - tpr_strict(LABELS, LR4_DEV),
                     "95% CI": f"[{lo:+.3f}, {hi:+.3f}]", "P(Δ>0)": float(np.mean(d > 0))})
dev_tab = pd.DataFrame(dev_rows)
print("\nE-3 real dev (fit on shadow memberships only; dev labels for evaluation only):")
print(dev_tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nCoefficients:", json.dumps({k: v["standardized_w"] for k, v in dev_coefs.items()}, indent=1))

np.savez(ENS_CACHE / "ens_dev_scores.npz", labels=LABELS, dev_order_sha=np.array(DEV_ORDER_SHA),
         **{fs.replace("+", "_plus_"): s for fs, s in dev_scores.items()})
dev_tab.to_csv(ENS_CACHE / "ens_dev_results.csv", index=False)
with open(ENS_CACHE / "ens_notes.json", "w") as f:
    json.dump({"models": FROZEN_NAMES,
               "features": {"LR": "frozen LR4 formula, κ=8", "B_ext": "mean-loss gap × √T over LR's eligible OUT rows",
                            "RMIA_a": "continuous a(x), mean-loss statistic, paired refs, trim int(0.2·K) per side"},
               "classifier": f"StandardScaler (training rows only) + L2 logistic regression, C={C_FIXED} fixed",
               "E2": "held-out pair removed from training rows and from every feature's reference pool; "
                     "training mocks exclude {a,b,m,partner}; held-out targets exclude {a,b}; 2 global text-group folds",
               "E3": "fit on 8 mocks (exclude m, partner) × all dev texts; real target uses all 8 paired + 4 fixed; "
                     "new-model transfer on known candidate texts", "EZ": "not included",
               "coefficients_E3": dev_coefs, "validation_submission": False}, f, indent=2)
print("Saved to", ENS_CACHE)

           role target  LR refs IN refs min–max OUT refs min–max  RMIA refs  RMIA trimmed per side
  training mock    inA       10             3–3              7–7          6                      1
  training mock    inB       10             3–3              7–7          6                      1
  training mock    p2A       10             3–4              6–7          6                      1
  training mock    p2B       10             3–4              6–7          6                      1
  training mock    p3A       10             3–4              6–7          6                      1
  training mock    p3B       10             3–4              6–7          6                      1
  training mock    p4A       10             3–4              6–7          6                      1
  training mock    p4B       10             3–4              6–7          6                      1
real dev target    dev       12             4–5              7–8          8                      1

E-3 real 

In [8]:
# G-0 — G-Drift pilot setup. After N-1..N-4 (GPU). Adapters are read only.
import time, math
GD_CACHE = DRIVE / "cache" / "gdrift"; GD_CACHE.mkdir(parents=True, exist_ok=True)
GD_REPO_COMMIT = "e34298fc40b1ff9e793a2f300140bec140f225eb"
LR_STEP, PROJ_SEED, HIDDEN = 1e-2, 20261019, 768
PILOT_ALL = np.random.default_rng(20261019).choice(2000, 200, replace=False)   # fixed random order
PILOT_N = 200                                     # stage 1; set to 200 only after stage 1 passes
PILOT_IDX = PILOT_ALL[:PILOT_N]
PILOT_MODELS = {"dev_target": ADAPTER_DIR, "p4A": DRIVE / "models" / "shadow_p4A", "p4B": DRIVE / "models" / "shadow_p4B"}
CACHED_LOSS = {"dev_target": TGT["train"], "p4A": REF["p4A"]["train"], "p4B": REF["p4B"]["train"]}
CONFIGS = {"C1_published_form": dict(params="all", loss="last"),
           "C2_seq_allparams":  dict(params="all", loss="mean"),
           "C3_seq_lora_only":  dict(params="lora", loss="mean")}
FEATURE_NAMES = ["loss_before", "logit_before", "feat_before", "loss_after", "logit_after", "feat_after", "euclid_drift"]
LAST_NAMES = [f + "_lastpos" for f in FEATURE_NAMES]
DIAG_NAMES = ["grad_norm", "dloss_first_order", "n_tokens", "mean_loss_before", "n_selected_no_grad"]
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False

def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
ADAPTER_SHA = {k: sha(Path(v) / "adapter_model.safetensors") for k, v in PILOT_MODELS.items()}
PILOT_IDS = [tokenizer(TEXTS["train"][i], return_tensors="pt", truncation=True, max_length=MAX_LENGTH)["input_ids"][0]
             for i in PILOT_IDX]
assert all(ids.numel() - 1 == TGT["train"][i].size for ids, i in zip(PILOT_IDS, PILOT_IDX))

_g = torch.Generator().manual_seed(PROJ_SEED)
V = torch.randn(HIDDEN, generator=_g); V = (V / V.norm()).to(device)      # fixed random unit projection

def load_fp32(path):
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=torch.float32)
    m = PeftModel.from_pretrained(base, path, is_trainable=False)
    m.config.pad_token_id = tokenizer.pad_token_id
    assert m.config.n_embd == HIDDEN
    return m.to(device).eval()                                    # dropout off (departure from published train mode)

def parts(out, ids):
    logits, h = out.logits[0, :-1].float(), out.hidden_states[-1][0, :-1].float()   # final layer, after ln_f
    tgt = ids[1:]
    return F.cross_entropy(logits, tgt, reduction="none"), logits.gather(1, tgt[:, None])[:, 0], h @ V, h

def drift_one(model, ids, params, all_params, backup, loss_mode):
    x = ids[None].to(device)
    try:
        model.zero_grad(set_to_none=True)
        out = model(input_ids=x, output_hidden_states=True, use_cache=False)
        ce, zt, pr, h = parts(out, x[0])
        (ce.mean() if loss_mode == "mean" else ce[-1]).backward()
        with torch.no_grad():
            got = [p for p in params if p.grad is not None]
            if not got:
                raise RuntimeError("no selected parameter received a gradient")
            gsq = sum((p.grad.double() ** 2).sum() for p in got).item()
            for p in got:
                p.add_(p.grad, alpha=LR_STEP)                     # θ ← θ + η∇L (= published SGD on −L)
            out2 = model(input_ids=x, output_hidden_states=True, use_cache=False)
            ce2, zt2, pr2, h2 = parts(out2, x[0])
            ce, zt, pr, h = ce.detach(), zt.detach(), pr.detach(), h.detach()
            series = (ce, zt, pr, ce2, zt2, pr2, (h2 - h).norm(dim=1))       # order = FEATURE_NAMES
            agg = (lambda t: t.mean()) if loss_mode == "mean" else (lambda t: t[-1])
            r = {n: float(agg(t)) for n, t in zip(FEATURE_NAMES, series)}
            r |= {n: float(t[-1]) for n, t in zip(LAST_NAMES, series)}
            r |= {"grad_norm": math.sqrt(gsq), "dloss_first_order": LR_STEP * gsq, "n_tokens": int(ce.numel()),
                  "mean_loss_before": float(ce.mean()), "n_selected_no_grad": len(params) - len(got)}
    finally:
        with torch.no_grad():
            for p, b in zip(all_params, backup):
                p.copy_(b)                                         # exact restore, also on exceptions
        model.zero_grad(set_to_none=True)
    return r

def write_meta(mname, cname, cfg, status, extra=None):
    meta = {"status": status, "model": mname, "adapter_path": str(PILOT_MODELS[mname]),
            "adapter_sha256": ADAPTER_SHA[mname], "base_model": BASE_MODEL_NAME,
            "tokenizer": {"source": str(ADAPTER_DIR), "truncation": True, "max_length": MAX_LENGTH,
                          "special_tokens": "GPT-2 default (none added)", "batch": 1, "padding": "none"},
            "dtype": "float32", "tf32": False, "mode": "eval (dropout off)", "config": cname, **cfg,
            "step": "single SGD ascent step, θ ← θ + η∇L, no momentum/weight decay", "lr": LR_STEP,
            "projection": {"seed": PROJ_SEED, "dim": HIDDEN, "unit_norm": True},
            "hidden_layer": "hidden_states[-1] (after ln_f)", "feature_names": FEATURE_NAMES,
            "lastpos_names": LAST_NAMES, "diag_names": DIAG_NAMES, "dev_idx": PILOT_IDX.tolist(),
            "pilot_n": len(PILOT_IDX), "repo_commit": GD_REPO_COMMIT, "paper": "arXiv 2604.00419v1"} | (extra or {})
    with open(GD_CACHE / f"pilot{len(PILOT_IDX)}_{mname}_{cname}.json", "w") as f:
        json.dump(meta, f, indent=1)
print(f"Pilot: {len(PILOT_IDX)} texts × {len(PILOT_MODELS)} models × {len(CONFIGS)} configs")

Pilot: 200 texts × 3 models × 3 configs


In [9]:
# G-1 — Feasibility pilot. No TPR/AUC; dev labels are not read.
N = len(PILOT_IDX)
rows, fails = [], []
for mname, mpath in PILOT_MODELS.items():
    model = load_fp32(mpath)
    all_params = list(model.parameters())
    backup = [p.detach().clone() for p in all_params]
    probe = PILOT_IDS[0][None].to(device)
    with torch.no_grad():
        ref_logits = model(input_ids=probe, use_cache=False).logits.clone()

    for cname, cfg in CONFIGS.items():
        write_meta(mname, cname, cfg, "running")
        for n, p in model.named_parameters():
            p.requires_grad_(cfg["params"] == "all" or "lora_" in n)
        params = [p for p in all_params if p.requires_grad]
        n_upd = sum(p.numel() for p in params); assert n_upd > 0
        feats, times, cfails = [], [], []
        for j, (ids, i) in enumerate(zip(PILOT_IDS, PILOT_IDX)):
            torch.cuda.synchronize(); t0 = time.perf_counter()
            try:
                r = drift_one(model, ids, params, all_params, backup, cfg["loss"])
            except Exception as e:
                cfails.append({"dev_idx": int(i), "error": repr(e)}); r = None
            torch.cuda.synchronize(); times.append(time.perf_counter() - t0)
            feats.append(r)
            if j % 25 == 0 or j == N - 1:
                with torch.no_grad():
                    d = (model(input_ids=probe, use_cache=False).logits - ref_logits).abs().max().item()
                assert d == 0.0, f"{mname}/{cname}: restored outputs differ by {d}"
        fails += [(mname, cname, f["dev_idx"], f["error"]) for f in cfails]
        ok = [r for r in feats if r is not None]
        keys = FEATURE_NAMES + LAST_NAMES + DIAG_NAMES
        np.savez(GD_CACHE / f"pilot{N}_{mname}_{cname}.npz", dev_idx=PILOT_IDX, dev_order_sha=np.array(DEV_ORDER_SHA),
                 ok=np.array([r is not None for r in feats]),
                 **{k: np.array([r[k] if r is not None else np.nan for r in feats], dtype=np.float64) for k in keys})
        if not ok:
            write_meta(mname, cname, cfg, "FAILED", {"failures": cfails})
            rows.append({"model": mname, "config": cname, "status": "FAILED", "ok": 0, "failed": N})
            continue

        r2 = drift_one(model, PILOT_IDS[0], params, all_params, backup, cfg["loss"]) if feats[0] else None
        rep = max(abs(r2[k] - feats[0][k]) for k in keys) if r2 else np.nan
        D = pd.DataFrame(ok)
        dl = D.loss_after - D.loss_before
        cache_diff = np.abs(D.mean_loss_before.to_numpy() -
                            np.array([CACHED_LOSS[mname][i].mean() for i, r in zip(PILOT_IDX, feats) if r is not None]))
        rows.append({"model": mname, "config": cname, "status": "ok", "updated params": n_upd, "ok": len(ok),
                     "failed": N - len(ok), "non-finite rows": int((~np.isfinite(D[keys].to_numpy())).any(1).sum()),
                     "max params w/o grad": int(D.n_selected_no_grad.max()), "ms/text": 1000 * np.median(times),
                     "grad norm med": D.grad_norm.median(), "grad norm max": D.grad_norm.max(),
                     "Δloss med": dl.median(), "Δloss max": dl.max(), "frac Δloss>0": float((dl > 0).mean()),
                     "Δloss / η‖g‖² med": float((dl / D.dloss_first_order).median()),
                     "drift med": D.euclid_drift.median(), "drift max": D.euclid_drift.max(),
                     "|fp32−cached| max": float(cache_diff.max()), "repeat max diff": rep})
        write_meta(mname, cname, cfg, "done", {"n_ok": len(ok), "failures": cfails,
                                                "median_ms_per_text": float(1000 * np.median(times))})
    for p in all_params:
        p.requires_grad_(False)
    with torch.no_grad():
        assert all(torch.equal(p, b) for p, b in zip(all_params, backup)), f"{mname}: parameters not restored"
    assert sha(Path(mpath) / "adapter_model.safetensors") == ADAPTER_SHA[mname], f"{mname}: adapter file changed"
    model.to("cpu"); del model, backup, all_params; gc.collect(); torch.cuda.empty_cache()

pilot = pd.DataFrame(rows)
pd.set_option("display.width", 280)
print(pilot.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print(f"Failures: {len(fails)}", fails[:5])
pilot.to_csv(GD_CACHE / f"pilot{N}_feasibility.csv", index=False)

     model            config status  updated params  ok  failed  non-finite rows  max params w/o grad  ms/text  grad norm med  grad norm max  Δloss med  Δloss max  frac Δloss>0  Δloss / η‖g‖² med  drift med  drift max  |fp32−cached| max  repeat max diff
dev_target C1_published_form     ok       129158400 200       0                0                    0    32.83          51.49          225.2      48.66      400.1          0.99              2.329      59.83      384.9            0.00403                0
dev_target  C2_seq_allparams     ok       129158400 200       0                0                    0    32.39          11.07          26.87      1.488      10.38             1               1.18      3.817      19.25            0.00403                0
dev_target  C3_seq_lora_only     ok         4718592 200       0                0                    0    25.49          1.434          4.408    0.02068     0.1984             1              1.006     0.3581      4.828            0.00403  

In [10]:
# GF-1 — Full G-Drift features: dev target + 8 frozen paired shadows × 3 configs × 2,000 dev texts.
# Reuses a cache only if its metadata matches the current run exactly; stale caches are renamed, never overwritten.
FULL_MODELS = {"dev_target": ADAPTER_DIR} | {n: DRIVE / "models" / f"shadow_{n}"
                                             for n in ["inA", "inB", "p2A", "p2B", "p3A", "p3B", "p4A", "p4B"]}
ALL_IDS = [tokenizer(t, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)["input_ids"][0] for t in TEXTS["train"]]
assert all(ids.numel() - 1 == t.size for ids, t in zip(ALL_IDS, TGT["train"]))
KEYS = FEATURE_NAMES + LAST_NAMES + DIAG_NAMES
TOK_SETTINGS = {"source": str(ADAPTER_DIR), "truncation": True, "max_length": MAX_LENGTH, "batch": 1,
                "padding": "none", "special_tokens": "GPT-2 default (none added)"}
fpath = lambda m, c, ext: GD_CACHE / f"full_{m}_{c}.{ext}"

def expected_meta(m, adsha, c):
    return {"model": m, "adapter_sha256": adsha, "base_model": BASE_MODEL_NAME, "dev_order_sha": DEV_ORDER_SHA,
            "tokenizer": TOK_SETTINGS, "dtype": "float32", "tf32": False, "mode": "eval (dropout off)",
            "params": CONFIGS[c]["params"], "loss": CONFIGS[c]["loss"], "lr": LR_STEP,
            "projection": {"seed": PROJ_SEED, "dim": HIDDEN, "unit_norm": True}, "hidden_layer": "hidden_states[-1]",
            "feature_names": FEATURE_NAMES, "lastpos_names": LAST_NAMES, "diag_names": DIAG_NAMES,
            "repo_commit": GD_REPO_COMMIT}

def write_full_meta(m, mpath, adsha, c, status, extra=None):
    meta = {"status": status, "config": c, "adapter_path": str(mpath),
            "step": "single SGD ascent step θ ← θ + η∇L", "texts": "all 2,000 dev texts in original order"} \
           | expected_meta(m, adsha, c) | (extra or {})
    with open(fpath(m, c, "json"), "w") as f:
        json.dump(meta, f, indent=1)

def cache_valid(m, c, adsha, verbose=True):
    pj, pz = fpath(m, c, "json"), fpath(m, c, "npz")
    if not (pj.exists() and pz.exists()):
        return False
    meta = json.load(open(pj))
    bad = [k for k, v in expected_meta(m, adsha, c).items() if meta.get(k) != v]
    if meta.get("status") != "done": bad.append("status")
    if np.load(pz)["dev_order_sha"].item() != DEV_ORDER_SHA: bad.append("npz dev_order_sha")
    if bad and verbose: print(f"{m}/{c}: cache mismatch on {bad}")
    return not bad

def retire(m, c):
    stamp = time.strftime("%Y%m%d_%H%M%S")
    for ext in ("npz", "json"):
        p = fpath(m, c, ext)
        if p.exists(): p.rename(p.with_name(p.name + f".stale_{stamp}"))

ADSHA = {m: sha(Path(p) / "adapter_model.safetensors") for m, p in FULL_MODELS.items()}
t_all = time.perf_counter()
for mname, mpath in FULL_MODELS.items():
    adsha = ADSHA[mname]
    todo = [c for c in CONFIGS if not cache_valid(mname, c, adsha)]
    if not todo:
        print(f"{mname}: all configs cached and verified"); continue
    for c in todo:
        retire(mname, c)
    model = load_fp32(mpath)
    all_params = list(model.parameters()); backup = [p.detach().clone() for p in all_params]
    probe = ALL_IDS[0][None].to(device)
    with torch.no_grad():
        ref_logits = model(input_ids=probe, use_cache=False).logits.clone()
    for cname in todo:
        cfg = CONFIGS[cname]
        write_full_meta(mname, mpath, adsha, cname, "running")
        for n, p in model.named_parameters():
            p.requires_grad_(cfg["params"] == "all" or "lora_" in n)
        params = [p for p in all_params if p.requires_grad]
        feats, cfails, t0 = [], [], time.perf_counter()
        for j, ids in enumerate(tqdm(ALL_IDS, desc=f"{mname}/{cname}", leave=False)):
            try:
                feats.append(drift_one(model, ids, params, all_params, backup, cfg["loss"]))
            except Exception as e:
                feats.append(None); cfails.append({"dev_idx": j, "error": repr(e)})
            if j % 250 == 0 or j == len(ALL_IDS) - 1:
                with torch.no_grad():
                    d = (model(input_ids=probe, use_cache=False).logits - ref_logits).abs().max().item()
                assert d == 0.0, f"{mname}/{cname}: restored outputs differ by {d}"
        np.savez(fpath(mname, cname, "npz"), dev_order_sha=np.array(DEV_ORDER_SHA),
                 ok=np.array([r is not None for r in feats]),
                 **{k: np.array([r[k] if r is not None else np.nan for r in feats], dtype=np.float64) for k in KEYS})
        write_full_meta(mname, mpath, adsha, cname, "done" if not cfails else "partial",
                        {"failures": cfails, "minutes": (time.perf_counter() - t0) / 60})
        print(f"{mname}/{cname}: {len(feats) - len(cfails)}/2000 ok, {(time.perf_counter() - t0) / 60:.1f} min")
    for p in all_params:
        p.requires_grad_(False)
    with torch.no_grad():
        assert all(torch.equal(p, b) for p, b in zip(all_params, backup))
    assert sha(Path(mpath) / "adapter_model.safetensors") == adsha, f"{mname}: adapter file changed"
    model.to("cpu"); del model, backup, all_params; gc.collect(); torch.cuda.empty_cache()
print(f"Total {(time.perf_counter() - t_all) / 60:.1f} min")

dev_target/C1_published_form:   0%|          | 0/2000 [00:00<?, ?it/s]

dev_target/C1_published_form: 2000/2000 ok, 1.1 min


dev_target/C2_seq_allparams:   0%|          | 0/2000 [00:00<?, ?it/s]

dev_target/C2_seq_allparams: 2000/2000 ok, 1.1 min


dev_target/C3_seq_lora_only:   0%|          | 0/2000 [00:00<?, ?it/s]

...

p4B/C1_published_form: 2000/2000 ok, 1.1 min


p4B/C2_seq_allparams:   0%|          | 0/2000 [00:00<?, ?it/s]

p4B/C2_seq_allparams: 2000/2000 ok, 1.1 min


p4B/C3_seq_lora_only:   0%|          | 0/2000 [00:00<?, ?it/s]

p4B/C3_seq_lora_only: 2000/2000 ok, 0.9 min
Total 28.1 min


In [13]:
# GF-2 — LR vs G-Drift vs combinations. Held-out pairs (features AND rows) × held-out text groups; fixed C; train-only scaling.
import joblib, sklearn
GD7 = list(FEATURE_NAMES)
GDF = {}
for m in ["dev_target"] + FROZEN_NAMES:
    for c in CONFIGS:
        assert cache_valid(m, c, ADSHA[m]), f"{m}/{c}: cache missing or stale; rerun GF-1"
        z = np.load(fpath(m, c, "npz"))
        assert z["ok"].shape == (2000,) and z["ok"].all(), (m, c)
        for k in KEYS:
            assert z[k].shape == (2000,) and np.isfinite(z[k]).all(), f"{m}/{c}/{k}: non-finite or wrong length"
        GDF[(m, c)] = {k: z[k] for k in GD7 + ["grad_norm"]}
for m in FROZEN_NAMES:
    for d in [MOCK for MOCK in [FOLD_FEAT[p][r] for p in FOLD_FEAT for r in ("train", "test")] if m in MOCK]:
        assert all(np.isfinite(v).all() and v.shape == (2000,) for v in d[m].values())
assert all(np.isfinite(v).all() and v.shape == (2000,) for v in REAL.values())

SETS = {"LR": ["LR"], "GD7 (drift alone)": GD7, "LR+GD7": ["LR"] + GD7,
        "LR+gn": ["LR", "grad_norm"], "LR+gn+GD7": ["LR", "grad_norm"] + GD7}
PRIMARY = "LR+gn+GD7"                                             # selection set, fixed in advance

dev_root = np.array([_find(k) for k in range(2000)])
perm = np.random.default_rng(20261018).permutation(np.unique(dev_root))   # same text folds as E-2
fold_of = {g: j % 2 for j, g in enumerate(perm)}
TFOLD = np.array([fold_of[g] for g in dev_root])

def fit(X, y):
    sc = StandardScaler().fit(X)
    return sc, LogisticRegression(C=C_FIXED, max_iter=5000).fit(sc.transform(X), y)
def design(ff, models, idx, cols):
    X = np.vstack([np.column_stack([ff(m)[c][idx] for c in cols]) for m in models])
    return X, np.concatenate([MEMD[m][idx].astype(int) for m in models])
def group_balanced(df, col):
    return 0.5 * df[df.aux == "fresh"][col].mean() + 0.5 * df[df.aux == "pooled"][col].mean()

def run_cv(cfg):
    rows, draws = [], {}
    for p, (a, b) in enumerate(FROZEN_PAIRS):
        ftr = lambda m: FOLD_FEAT[p]["train"][m] | GDF[(m, cfg)]
        fte = lambda m: FOLD_FEAT[p]["test"][m] | GDF[(m, cfg)]
        trm = [n for n in FROZEN_NAMES if n not in (a, b)]
        for f in range(2):
            tr, te = np.flatnonzero(TFOLD != f), np.flatnonzero(TFOLD == f)
            fitted = {s: (*fit(*design(ftr, trm, tr, cols)), cols) for s, cols in SETS.items()}
            for m in (a, b):
                y = MEMD[m][te].astype(int)
                pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
                rng = np.random.default_rng(1000 + 10 * FROZEN_NAMES.index(m) + f)
                bidx = [np.concatenate([rng.choice(pos, pos.size), rng.choice(neg, neg.size)]) for _ in range(N_BOOT)]
                for s, (sc, clf, cols) in fitted.items():
                    v = clf.decision_function(sc.transform(design(fte, [m], te, cols)[0]))
                    draws[(m, f, s)] = np.array([strict_fast(v[ix], y[ix]) for ix in bidx])
                    rows.append({"config": cfg, "target": m, "aux": "fresh" if m in FRESH_AUX else "pooled",
                                 "fold": f, "set": s, "TPR": tpr_strict(y, v), "AUC": roc_auc_score(y, v)})
    return pd.DataFrame(rows), draws

def summarize(cv, draws):
    out = []
    for m in FROZEN_NAMES:
        for s in SETS:
            sub = cv[(cv.target == m) & (cv.set == s)]
            r = {"target": m, "aux": "fresh" if m in FRESH_AUX else "pooled", "set": s, "TPR": sub.TPR.mean(), "AUC": sub.AUC.mean()}
            for base in ("LR", "LR+gn"):
                d = np.mean([draws[(m, f, s)] - draws[(m, f, base)] for f in range(2)], axis=0)
                lo, hi = np.percentile(d, [2.5, 97.5])
                r[f"Δ vs {base}"] = sub.TPR.mean() - cv[(cv.target == m) & (cv.set == base)].TPR.mean()
                r[f"CI vs {base}"] = f"[{lo:+.3f}, {hi:+.3f}]"
            out.append(r)
    return pd.DataFrame(out)

ALL_CV, SUMM = {}, {}
for cfg in CONFIGS:
    cv, draws = run_cv(cfg); ALL_CV[cfg] = cv; SUMM[cfg] = summarize(cv, draws)

ov = []
for c in CONFIGS:
    for s in SETS:
        d = SUMM[c][SUMM[c].set == s]
        r = {"config": c, "set": s,
             "TPR fresh": d[d.aux == "fresh"].TPR.mean(), "TPR pooled": d[d.aux == "pooled"].TPR.mean(),
             "TPR balanced": group_balanced(d, "TPR"), "AUC balanced": group_balanced(d, "AUC"),
             "Δ vs LR (bal)": group_balanced(d, "Δ vs LR"), "Δ vs LR+gn (bal)": group_balanced(d, "Δ vs LR+gn")}
        r["beats BOTH LR and LR+gn"] = bool(r["Δ vs LR (bal)"] > 0 and r["Δ vs LR+gn (bal)"] > 0) if "GD7" in s else "-"
        ov.append(r)
overview = pd.DataFrame(ov)
pd.set_option("display.width", 260)
print("Held-out mock targets (fresh and pooled groups weighted equally):")
print(overview.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

sel_score = {c: float(overview[(overview.config == c) & (overview.set == PRIMARY)]["TPR balanced"].iloc[0]) for c in CONFIGS}
SEL = max(CONFIGS, key=lambda c: (round(sel_score[c], 6), c == "C1_published_form"))
print(f"\nConfig selected on shadow tasks (group-balanced held-out TPR of {PRIMARY}): {SEL} | {sel_score}")
print(f"\nPer held-out target, {SEL}:")
print(SUMM[SEL].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Real dev, selected config only. Fit on 8 mocks (refs exclude target+partner) × all dev texts; shadow labels only.
MOCK_FULL = {m: features(ML[m], {m, PARTNER[m]})[0] for m in FROZEN_NAMES}
ff_tr = lambda m: MOCK_FULL[m] | GDF[(m, SEL)]
real = REAL | GDF[("dev_target", SEL)]
MODEL_DIR = GD_CACHE / f"frozen_classifiers_{SEL}"; MODEL_DIR.mkdir(exist_ok=True)
dev_scores, dev_rows, frozen = {}, [], {}
for s, cols in SETS.items():
    sc, clf = fit(*design(ff_tr, FROZEN_NAMES, np.arange(2000), cols))
    dev_scores[s] = clf.decision_function(sc.transform(np.column_stack([real[c] for c in cols])))
    tag = s.replace("+", "_plus_").replace(" ", "_").replace("(", "").replace(")", "")
    joblib.dump({"scaler": sc, "classifier": clf, "feature_order": cols, "config": SEL}, MODEL_DIR / f"{tag}.joblib")
    frozen[s] = {"feature_order": cols, "scaler_mean": sc.mean_.tolist(), "scaler_scale": sc.scale_.tolist(),
                 "coef": clf.coef_[0].tolist(), "intercept": float(clf.intercept_[0]), "C": C_FIXED,
                 "training": "8 frozen paired shadows (refs exclude target+partner) × 2,000 dev texts, shadow labels",
                 "sklearn": sklearn.__version__, "file": f"{tag}.joblib"}
json.dump(frozen, open(MODEL_DIR / "frozen_classifiers.json", "w"), indent=1)
assert np.isclose(tpr_strict(LABELS, dev_scores["LR"]), tpr_strict(LABELS, LR4_DEV))

bt = {s: np.array([strict_fast(v[ix], LABELS[ix]) for ix in ENS_BOOT]) for s, v in dev_scores.items()}
for s, v in dev_scores.items():
    r = {"set": s, "dev AUC": roc_auc_score(LABELS, v), "strict TPR": tpr_strict(LABELS, v)}
    for base in ("LR", "LR+gn"):
        d = bt[s] - bt[base]; lo, hi = np.percentile(d, [2.5, 97.5])
        r[f"Δ vs {base}"] = r["strict TPR"] - tpr_strict(LABELS, dev_scores[base]); r[f"CI vs {base}"] = f"[{lo:+.3f}, {hi:+.3f}]"
    dev_rows.append(r)
dev_tab = pd.DataFrame(dev_rows)
print(f"\nReal dev target ({SEL}; shadow-label fit; new model on known candidate texts):")
print(dev_tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("Coefficients:", json.dumps({s: dict(zip(v["feature_order"], np.round(v["coef"], 3).tolist())) for s, v in frozen.items()}, indent=1))

for c in CONFIGS:
    ALL_CV[c].to_csv(GD_CACHE / f"cv_rows_{c}.csv", index=False); SUMM[c].to_csv(GD_CACHE / f"cv_summary_{c}.csv", index=False)
overview.to_csv(GD_CACHE / "cv_overview.csv", index=False); dev_tab.to_csv(GD_CACHE / f"dev_results_{SEL}.csv", index=False)
np.savez(GD_CACHE / f"dev_scores_{SEL}.npz", labels=LABELS, dev_order_sha=np.array(DEV_ORDER_SHA),
         **{s.replace("+", "_plus_").replace(" ", "_").replace("(", "").replace(")", ""): v for s, v in dev_scores.items()})
json.dump({"sets": SETS, "primary_selection_set": PRIMARY, "selected_config": SEL, "selection_scores": sel_score,
           "selection_rule": "group-balanced (fresh 50% / pooled 50%) held-out mock strict TPR of primary set; tie -> C1",
           "improvement_rule": "group-balanced Δ > 0 vs BOTH LR and LR+gn", "C": C_FIXED,
           "protocol": "fold-specific LR refs (E-1); held-out pair excluded from rows and references; 2 text-group folds",
           "frozen_classifiers": str(MODEL_DIR), "validation_submission": False},
          open(GD_CACHE / "gdrift_eval_notes.json", "w"), indent=2)
print("Saved to", GD_CACHE)

[... log truncated ...]
2   0.0071 [-0.025, +0.018]      0.0105 [-0.012, +0.022]
   p3B pooled                LR 0.8109 0.9835   0.0000 [+0.000, +0.000]     -0.0098 [-0.015, +0.009]
   p3B pooled GD7 (drift alone) 0.0137 0.5474  -0.7972 [-0.875, -0.737]     -0.8070 [-0.877, -0.736]
   p3B pooled            LR+GD7 0.8257 0.9834   0.0148 [-0.014, +0.028]      0.0050 [-0.015, +0.023]
   p3B pooled             LR+gn 0.8207 0.9836   0.0098 [-0.009, +0.015]      0.0000 [+0.000, +0.000]
   p3B pooled         LR+gn+GD7 0.8218 0.9833   0.0109 [-0.019, +0.030]      0.0011 [-0.018, +0.024]
   p4A  fresh                LR 0.7863 0.9799   0.0000 [+0.000, +0.000]      0.0070 [-0.013, +0.020]
   p4A  fresh GD7 (drift alone) 0.0200 0.5324  -0.7663 [-0.835, -0.648]     -0.7592 [-0.832, -0.649]
   p4A  fresh            LR+GD7 0.7822 0.9797  -0.0040 [-0.022, +0.013]      0.0030 [-0.018, +0.019]
   p4A  fresh             LR+gn 0.7793 0.9798  -0.0070 [-0.020, +0.013]      0.0000 [+0.000, +0.000]
   p4A  fr

In [14]:
# CD-1 — Per-text calibrated drift (C2), shared-variance Gaussian LR (variant B, stable form), κ=8.
# Pooled variance estimated on training text folds only. No new inference.
import joblib, sklearn
CD_CFG, KAPPA_CD = "C2_seq_allparams", 8
CD_DIR = GD_CACHE / "calibrated_drift"; CD_DIR.mkdir(exist_ok=True)
MV = {"dloss": {m: GDF[(m, CD_CFG)]["loss_after"] - GDF[(m, CD_CFG)]["loss_before"] for m in ["dev_target"] + FROZEN_NAMES},
      "drift": {m: GDF[(m, CD_CFG)]["euclid_drift"].copy() for m in ["dev_target"] + FROZEN_NAMES}}
for k in MV:
    assert all(v.shape == (2000,) and np.isfinite(v).all() for v in MV[k].values()), k
ALL_TXT = np.arange(2000)

def shared_lr(t, R, Rin, pool_idx=None, pool=None, kappa=KAPPA_CD):
    """Variant B. Per-text mean/residuals from references; pooled variance from pool_idx texts (or given)."""
    Rout = ~Rin
    n_in, n_out = Rin.sum(0), Rout.sum(0)
    assert (n_in > 0).all() and (n_out > 0).all()
    mu_in = np.where(Rin, R, 0.0).sum(0) / n_in
    mu_out = np.where(Rout, R, 0.0).sum(0) / n_out
    rss = np.where(Rin, (R - mu_in) ** 2, 0.0).sum(0) + np.where(Rout, (R - mu_out) ** 2, 0.0).sum(0)
    df = np.maximum(n_in - 1, 0) + np.maximum(n_out - 1, 0)
    if pool is None:
        assert df[pool_idx].sum() > 0
        pool = rss[pool_idx].sum() / df[pool_idx].sum()
    assert pool > 0
    v = (rss + kappa * pool) / (df + kappa)
    assert (v > 0).all()
    s = (mu_in - mu_out) * (t - 0.5 * (mu_in + mu_out)) / v          # = ((t−μo)² − (t−μi)²) / (2v)
    assert np.isfinite(s).all()
    return s, float(pool), (n_in, n_out, df), (mu_in, mu_out, v)

# One-time check: stable form equals the squared-difference form.
_R = np.stack([MV["drift"][n] for n in FROZEN_NAMES]); _Rin = np.stack([MEMD[n] for n in FROZEN_NAMES])
_s, _, _, (_mi, _mo, _v) = shared_lr(MV["drift"]["dev_target"], _R, _Rin, pool_idx=ALL_TXT)
_t = MV["drift"]["dev_target"]
assert np.allclose(_s, ((_t - _mo) ** 2 - (_t - _mi) ** 2) / (2 * _v), rtol=1e-9, atol=1e-9)

def cal_feats(tname, excl, pool_idx=None, pools=None):
    refs = [n for n in FROZEN_NAMES if n not in set(excl)]
    Rin = np.stack([MEMD[n] for n in refs])
    out, used_pools = {}, {}
    for k in MV:
        out[f"cal_{k}"], used_pools[k], (n_in, n_out, df), _ = shared_lr(
            MV[k][tname], np.stack([MV[k][n] for n in refs]), Rin,
            pool_idx=pool_idx, pool=None if pools is None else pools[k])
    info = {"drift refs": len(refs), "IN min–max": f"{n_in.min()}–{n_in.max()}",
            "OUT min–max": f"{n_out.min()}–{n_out.max()}", "shared df min–max": f"{df.min()}–{df.max()}"}
    return out, info, set(refs), used_pools

# Fold-specific features: reference exclusions as in E-1; pooled variance from the training text fold only.
CAL, crow = {}, []
for p, (a, b) in enumerate(FROZEN_PAIRS):
    for f in range(2):
        tr = np.flatnonzero(TFOLD != f)
        CAL[(p, f)] = {"train": {}, "test": {}}
        for m in [n for n in FROZEN_NAMES if n not in (a, b)]:
            CAL[(p, f)]["train"][m], info, used, _ = cal_feats(m, {a, b, m, PARTNER[m]}, pool_idx=tr)
            assert not used & {a, b, m, PARTNER[m]}
            if f == 0: crow.append({"held-out pair": f"{a}/{b}", "role": "training mock", "target": m, **info})
        for m in (a, b):
            CAL[(p, f)]["test"][m], info, used, _ = cal_feats(m, {a, b}, pool_idx=tr)
            assert not used & {a, b}
            if f == 0: crow.append({"held-out pair": f"{a}/{b}", "role": "held-out target", "target": m, **info})

# Frozen-fit features: pooled over all dev texts (no labels); pools saved for later scoring.
CAL_FULL, FROZEN_POOLS = {}, {}
for m in FROZEN_NAMES:
    CAL_FULL[m], info, used, FROZEN_POOLS[f"mock {m} (excl. {m},{PARTNER[m]})"] = cal_feats(m, {m, PARTNER[m]}, pool_idx=ALL_TXT)
    assert not used & {m, PARTNER[m]}
    crow.append({"held-out pair": "-", "role": "frozen-fit mock", "target": m, **info})
CAL_REAL, info, _, FROZEN_POOLS["all 8 frozen refs (real target, p5/p6)"] = cal_feats("dev_target", (), pool_idx=ALL_TXT)
crow.append({"held-out pair": "-", "role": "real dev target", "target": "dev", **info})
json.dump({"kappa": KAPPA_CD, "pooled_variance_texts": "all 2,000 dev texts", "pools": FROZEN_POOLS},
          open(CD_DIR / "frozen_pooled_variances.json", "w"), indent=1)

counts = pd.DataFrame(crow)
print("Drift-calibration reference counts (paired shadows only):")
print(counts.groupby("role")[["drift refs", "IN min–max", "OUT min–max", "shared df min–max"]]
      .agg(lambda s: sorted(set(s))).to_string())
counts.to_csv(CD_DIR / "reference_counts.csv", index=False)

CD_SETS = {"LR": ["LR"], "LR+cal": ["LR", "cal_dloss", "cal_drift"],
           "cal_dloss alone [diag]": ["cal_dloss"], "cal_drift alone [diag]": ["cal_drift"]}
rows, draws = [], {}
for p, (a, b) in enumerate(FROZEN_PAIRS):
    trm = [n for n in FROZEN_NAMES if n not in (a, b)]
    for f in range(2):
        ftr = lambda m: FOLD_FEAT[p]["train"][m] | CAL[(p, f)]["train"][m]
        fte = lambda m: FOLD_FEAT[p]["test"][m] | CAL[(p, f)]["test"][m]
        tr, te = np.flatnonzero(TFOLD != f), np.flatnonzero(TFOLD == f)
        fitted = {s: (*fit(*design(ftr, trm, tr, cols)), cols) for s, cols in CD_SETS.items()}
        for m in (a, b):
            y = MEMD[m][te].astype(int)
            pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
            rng = np.random.default_rng(1000 + 10 * FROZEN_NAMES.index(m) + f)
            bidx = [np.concatenate([rng.choice(pos, pos.size), rng.choice(neg, neg.size)]) for _ in range(N_BOOT)]
            for s, (sc, clf, cols) in fitted.items():
                v = clf.decision_function(sc.transform(design(fte, [m], te, cols)[0]))
                draws[(m, f, s)] = np.array([strict_fast(v[ix], y[ix]) for ix in bidx])
                rows.append({"target": m, "aux": "fresh" if m in FRESH_AUX else "pooled", "fold": f, "set": s,
                             "TPR": tpr_strict(y, v), "AUC": roc_auc_score(y, v)})
cv = pd.DataFrame(rows)

summ = []
for m in FROZEN_NAMES:
    for s in CD_SETS:
        sub, base = cv[(cv.target == m) & (cv.set == s)], cv[(cv.target == m) & (cv.set == "LR")]
        d = np.mean([draws[(m, f, s)] - draws[(m, f, "LR")] for f in range(2)], axis=0)
        lo, hi = np.percentile(d, [2.5, 97.5])
        summ.append({"target": m, "aux": "fresh" if m in FRESH_AUX else "pooled", "set": s, "TPR": sub.TPR.mean(),
                     "AUC": sub.AUC.mean(), "Δ vs LR": sub.TPR.mean() - base.TPR.mean(), "CI vs LR": f"[{lo:+.3f}, {hi:+.3f}]"})
summ = pd.DataFrame(summ)
pd.set_option("display.width", 220)
print("\nPer held-out target:"); print(summ.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

ov = []
for s in CD_SETS:
    d = summ[summ.set == s]
    ov.append({"set": s, "TPR fresh": d[d.aux == "fresh"].TPR.mean(), "TPR pooled": d[d.aux == "pooled"].TPR.mean(),
               "TPR balanced": group_balanced(d, "TPR"), "AUC fresh": d[d.aux == "fresh"].AUC.mean(),
               "AUC pooled": d[d.aux == "pooled"].AUC.mean(), "Δ fresh": d[d.aux == "fresh"]["Δ vs LR"].mean(),
               "Δ pooled": d[d.aux == "pooled"]["Δ vs LR"].mean(), "Δ balanced": group_balanced(d, "Δ vs LR")})
ov = pd.DataFrame(ov)
print("\nGroup summary (fresh and pooled reported separately; balanced = 50/50):")
print(ov.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
r = ov[ov.set == "LR+cal"].iloc[0]
PROMISING = bool(r["Δ balanced"] > 0 and r["Δ fresh"] >= 0 and r["Δ pooled"] >= 0)
print(f"\nScreening rule (balanced Δ>0 and neither group negative): {'PASS → test frozen combination on p5/p6' if PROMISING else 'FAIL'}")

# Frozen combination: 8 mocks (refs exclude target+partner) × all dev texts; shadow labels only.
ff_full = lambda m: MOCK_FULL[m] | CAL_FULL[m]
frozen = {}
for s in ("LR", "LR+cal"):
    cols = CD_SETS[s]
    sc, clf = fit(*design(ff_full, FROZEN_NAMES, ALL_TXT, cols))
    tag = s.replace("+", "_plus_")
    joblib.dump({"scaler": sc, "classifier": clf, "feature_order": cols, "config": CD_CFG, "kappa": KAPPA_CD,
                 "pooled_variances_for_all8_refs": FROZEN_POOLS["all 8 frozen refs (real target, p5/p6)"]},
                CD_DIR / f"frozen_{tag}.joblib")
    frozen[s] = {"feature_order": cols, "scaler_mean": sc.mean_.tolist(), "scaler_scale": sc.scale_.tolist(),
                 "coef": clf.coef_[0].tolist(), "intercept": float(clf.intercept_[0]), "C": C_FIXED,
                 "sklearn": sklearn.__version__, "file": f"frozen_{tag}.joblib"}
json.dump(frozen, open(CD_DIR / "frozen_classifiers.json", "w"), indent=1)
print("\nFrozen LR+cal coefficients:", dict(zip(frozen["LR+cal"]["feature_order"], np.round(frozen["LR+cal"]["coef"], 3))))

if PROMISING:
    real = REAL | CAL_REAL
    dev = {}
    for s in ("LR", "LR+cal"):
        o = joblib.load(CD_DIR / f"frozen_{s.replace('+', '_plus_')}.joblib")
        dev[s] = o["classifier"].decision_function(o["scaler"].transform(np.column_stack([real[c] for c in o["feature_order"]])))
    assert np.isclose(tpr_strict(LABELS, dev["LR"]), tpr_strict(LABELS, LR4_DEV))
    d = np.array([strict_fast(dev["LR+cal"][ix], LABELS[ix]) - strict_fast(dev["LR"][ix], LABELS[ix]) for ix in ENS_BOOT])
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"\nReal dev (exploratory): LR {tpr_strict(LABELS, dev['LR']):.4f} | LR+cal {tpr_strict(LABELS, dev['LR+cal']):.4f} "
          f"(Δ {tpr_strict(LABELS, dev['LR+cal']) - tpr_strict(LABELS, dev['LR']):+.4f}, CI [{lo:+.3f}, {hi:+.3f}]) | "
          f"AUC {roc_auc_score(LABELS, dev['LR']):.4f} vs {roc_auc_score(LABELS, dev['LR+cal']):.4f}")
    np.savez(CD_DIR / "dev_scores.npz", labels=LABELS, dev_order_sha=np.array(DEV_ORDER_SHA),
             LR=dev["LR"], LR_plus_cal=dev["LR+cal"])
else:
    print("Screening rule failed: real dev target not scored.")

cv.to_csv(CD_DIR / "cv_rows.csv", index=False); summ.to_csv(CD_DIR / "cv_summary.csv", index=False)
ov.to_csv(CD_DIR / "cv_overview.csv", index=False)
json.dump({"config": CD_CFG, "measurements": {"dloss": "loss_after − loss_before (mean CE)", "drift": "euclid_drift (mean over positions)"},
           "scaling": "raw (no √T)", "score": "shared-variance Gaussian LR (variant B), stable form (μi−μo)(t−(μi+μo)/2)/v, κ=8",
           "pooled_variance": "CV: training text fold only, per reference configuration; frozen: all dev texts (saved)",
           "LR_feature": "frozen LR4 formula unchanged (its pooled variances use all scored texts, unlabeled)",
           "references": "paired shadows only; training mocks exclude {a,b,m,partner}; held-out exclude {a,b}; "
                         "frozen-fit mocks exclude {m,partner}; real/p5/p6 use all 8",
           "rule": "screening: balanced Δ>0 vs LR and neither group negative", "passed": PROMISING,
           "dev_scored": PROMISING, "validation_submission": False}, open(CD_DIR / "notes.json", "w"), indent=2)
print("Saved to", CD_DIR)

Drift-calibration reference counts (paired shadows only):
                drift refs  IN min–max OUT min–max shared df min–max
role                                                                
frozen-fit mock        [6]  [3–3, 3–4]  [2–3, 3–3]             [4–4]
held-out target        [6]  [3–3, 3–4]  [2–3, 3–3]             [4–4]
real dev target        [8]       [4–5]       [3–4]             [6–6]
training mock          [4]  [2–2, 2–3]  [1–2, 2–2]             [2–2]

Per held-out target:
target    aux                    set    TPR    AUC  Δ vs LR         CI vs LR
   inA pooled                     LR 0.8512 0.9857   0.0000 [+0.000, +0.000]
   inA pooled                 LR+cal 0.8531 0.9857   0.0020 [-0.006, +0.003]
   inA pooled cal_dloss alone [diag] 0.0070 0.4969  -0.8442 [-0.885, -0.768]
   inA pooled cal_drift alone [diag] 0.0348 0.6198  -0.8163 [-0.861, -0.734]
   inB pooled                     LR 0.8277 0.9861   0.0000 [+0.000, +0.000]
   inB pooled                 LR+cal 0.8277 